# Monitor Script

This is an example script monitoring values form the Enviro board and the external temperature sensor. I tried to organize the code in a way that makes it easy learn and to use parts of it for your scripts.

In [9]:
# Needed to communicate with the BME280 sensor (temperature, humidity, pressure)

from bme280 import BME280
import os
from dotenv import load_dotenv
bme280 = BME280()

import gspread
from oauth2client.service_account import ServiceAccountCredentials

sheet_name = "PiLog"   # Make sure this matches your Google sheet name

# use creds to create a client to interact with the Google Drive API
scopes = ["https://www.googleapis.com/auth/drive", "https://www.googleapis.com/auth/drive.file", 
          "https://www.googleapis.com/auth/spreadsheets"]
creds = ServiceAccountCredentials.from_json_keyfile_name('client_secret.json', scopes)
client = gspread.authorize(creds)

# Find the workbook by name and open the first sheet
sheet = client.open(sheet_name).sheet1


JSONDecodeError: Invalid control character at: line 2 column 82 (char 165)

In [2]:
# This is needed if you want to use the light sensor

from ltr559 import LTR559
ltr559 = LTR559()
load_dotenv()

True

In [3]:
# Added code to read the CCS811 sensor values

import busio
import board
import adafruit_ccs811

i2c = board.I2C() 
ccs = adafruit_ccs811.CCS811(i2c)

while (not ccs.data_ready):
    pass

print ("CCS811 Sensor ready.")

CCS811 Sensor ready.


In [4]:
# This is what we need to read the external temperature sensor

import smbus

# Simple helper routine to convert the reading to degrees C
def readTemp():
    # By default the address of LM75A is set to 0x48
    address = 0x48

    # Read I2C data and calculate temperature
    bus = smbus.SMBus(1)
    raw = bus.read_word_data(address, 0) & 0xFFFF
    raw = ((raw << 8) & 0xFF00) + (raw >> 8)
    temperature = (raw / 32.0) / 8.0
    return temperature

In [5]:
# This is what we need to communicate with the Adafruit Cloud

from Adafruit_IO import Client, Feed, RequestError
import os
ADAFRUIT_IO_USERNAME = "ThelmaDefish"
ADAFRUIT_IO_KEY = os.environ.get("ADAF")

aio = Client(ADAFRUIT_IO_USERNAME, ADAFRUIT_IO_KEY)

In [6]:
# Routines to write to the LCD screen

import ST7735
from PIL import Image, ImageDraw, ImageFont
from fonts.ttf import RobotoMedium as UserFont

disp = ST7735.ST7735(port=0,cs=1,dc=9,backlight=12,rotation=270,spi_speed_hz=10000000)
disp.begin()

back_color = (0,191,230)
text_color = (255, 255, 255)
head_color = (0,0,102)
warn_color = (255,  50,  50)
font_size  = 14
text_margin= 5

img = Image.new('RGBA', (disp.width, disp.height))
draw = ImageDraw.Draw(img)
font = ImageFont.truetype(UserFont, font_size)
font_big = ImageFont.truetype(UserFont, 2*font_size)
size_x, size_y = draw.textsize('text', font)

# We pass the name and the value so we can 'cycle' through the sensors
def updateLCD(label, value, unit): 
    if (value < 1000):
        vstr="{0:.1f} {1}".format(value, unit)
    else:
        vstr="{0:.0f} {1}".format(value, unit)
        
    draw.rectangle((0, 0, disp.width, disp.height), back_color)
    draw.text((text_margin, text_margin), "ORCSPICamp Station", font=font, fill=head_color)
    draw.text((text_margin, text_margin+2.0*size_y), label, font=font, fill=text_color)
    draw.text((text_margin, text_margin+3.5*size_y), vstr, font=font_big, fill=text_color)
    disp.display(img)


In [7]:
# Some general settings
import time
from datetime import datetime
from IPython.display import clear_output, display, update_display

# Get feeds using our routine
tempFeed  = aio.feeds("temperature")
temp2Feed = aio.feeds("tempexternal")
humidFeed = aio.feeds("humidity")
pressFeed = aio.feeds("pressure")
lightFeed = aio.feeds("light")
co2Feed   = aio.feeds("co2")
tvocFeed  = aio.feeds("tvoc")

# Set metadata associated with our measurement station. Update for your location
metadata = {'lat': 36.010357, 'lon': -84.269646, 'ele': 850, 'created_at': None}

## Main loop

In [ ]:
start_time   = time.time()
current_time = start_time

import pytz

tz = pytz.timezone('US/Eastern')


update_data = 20.0
update_lcd  = 3.0
next_lcd    = 't'

running = True

while running:
    try:        
        # Only do the data update if elapsed time is multiple of upodate_data
        if ((int (current_time - start_time) % update_data) == 0):
            p  = bme280.get_pressure()
            t  = bme280.get_temperature()
            h  = bme280.get_humidity()
            l  = ltr559.get_lux()
            c  = ccs.eco2
            v  = ccs.tvoc
            t2 = readTemp() 

            clear_output(wait=True)          

            out="T1: {0:.1f} C - T2: {1:.1f} C - Humidity: {2:.1f} % - Pressure: {3:.0f} hPa".format(t,t2,h,p)
            print(out)
            out="CO2: {0:.1f} ppm - TVOC: {1:.1f} ppb".format(c,v)
            print(out)
            print('Updated:',datetime.now(tz))

            f = (t * (9/5)) + 32
            f2 = (t2 * (9/5)) + 32
            aio.send_data(tempFeed.key,  f, metadata)
            aio.send_data(temp2Feed.key, f2, metadata)
            aio.send_data(humidFeed.key, h, metadata)
            aio.send_data(pressFeed.key, p, metadata)
            aio.send_data(lightFeed.key, l, metadata)
            aio.send_data(co2Feed.key, c, metadata)
            aio.send_data(tvocFeed.key, v, metadata)
        
        # We update the LCD every cycle
        if (next_lcd == 't'):
            f = (t * (9/5)) + 32
            updateLCD('Temperature', f, 'Fº')
            print('Temperature', f, 'Fº')
            next_lcd = 't2'
        elif (next_lcd == 't2'):
            f2 = (t2 * (9/5)) + 32
            updateLCD('Temperature (ext)', f2, 'Fº')
            print('Temperature (ext)', f2, 'Fº')
            next_lcd = 'h'
        elif (next_lcd == 'h'):
            updateLCD('Humidity', h, '%')
            print('Humidity', h, '%')
            next_lcd = 'p'
        elif (next_lcd == 'p'):
            updateLCD('Pressure', p, 'hPa')
            print('Pressure', p, 'hPa')
            next_lcd = 'c'
        elif (next_lcd == 'c'):
            updateLCD('CO2', c, 'ppm')
            print('CO2', c, 'ppm')
            next_lcd = 'v'
        elif (next_lcd == 'v'):
            updateLCD('TVOC', v, 'ppb')
            print('TVOC', v, 'ppb')
            next_lcd = 't'
            
        row = [str(datetime.now(tz)), str(t), str(h), str(c)]

        sheet.append_row(row)

        current_time = time.time()
        time.sleep(update_lcd)
        
    except IOError:
        pass
    except RuntimeError:
        pass


T1: 22.2 C - T2: 19.9 C - Humidity: 85.6 % - Pressure: 638 hPa
CO2: 405.0 ppm - TVOC: 0.0 ppb
Updated: 2021-10-20 08:39:05.236419-04:00
Temperature 72.02754992916365 Fº
Temperature (ext) 67.775 Fº
Humidity 85.55135448404721 %
Pressure 637.7370817742203 hPa
CO2 405 ppm
TVOC 0 ppb
Temperature 72.02754992916365 Fº
Temperature (ext) 67.775 Fº
Humidity 85.55135448404721 %
Pressure 637.7370817742203 hPa
CO2 405 ppm
TVOC 0 ppb
Temperature 72.02754992916365 Fº
